# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [2]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [3]:
links = fetch_website_links("https://edwarddonner.com")
links

['https://edwarddonner.com/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2025/11/11/ai-live-event/',
 'https://edwarddonner.com/2025/11/11/ai-live-event/',
 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-aws-at-scale/',
 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [4]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [5]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [6]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://edwarddonner.com/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/
https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/
https://edwarddonner.com/2025/11/11/

In [7]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [8]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'homepage', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'blog page', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'external partner page',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'},
  {'type': 'LinkedIn page', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'Twitter page', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'Facebook page',
   'url': 'https://www.facebook.com/edward.donner.52'}]}

In [9]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [10]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling gpt-5-nano
Found 9 relevant links


{'links': [{'type': 'homepage', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'portfolio', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'project', 'url': 'https://edwarddonner.com/connect-four/'},
  {'type': 'project', 'url': 'https://edwarddonner.com/outsmart/'},
  {'type': 'blog', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'linkedin', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'twitter', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'facebook', 'url': 'https://www.facebook.com/edward.donner.52'}]}

In [11]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 10 relevant links


{'links': [{'type': 'about page', 'url': 'https://huggingface.co/brand'},
  {'type': 'company page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'GitHub', 'url': 'https://github.com/huggingface'},
  {'type': 'Twitter', 'url': 'https://twitter.com/huggingface'},
  {'type': 'LinkedIn', 'url': 'https://www.linkedin.com/company/huggingface/'},
  {'type': 'Discussion forum', 'url': 'https://discuss.huggingface.co'},
  {'type': 'Status page', 'url': 'https://status.huggingface.co/'},
  {'type': 'Endpoints', 'url': 'https://endpoints.huggingface.co'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [12]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [13]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 11 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Community
Docs
Enterprise
Pricing
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
fal/Qwen-Image-Edit-2511-Multiple-Angles-LoRA
Updated
7 days ago
•
36.9k
•
635
Lightricks/LTX-2
Updated
about 9 hours ago
•
1.06M
•
1k
zai-org/GLM-Image
Updated
about 22 hours ago
•
203
•
544
nvidia/nemotron-speech-streaming-en-0.6b
Updated
9 days ago
•
3.9k
•
369
openbmb/AgentCPM-Explore
Updated
about 19 hours ago
•
77
•
266
Browse 2M+ models
Spaces
Running
on
Zero
Featured
722
Qwen Image Multiple Angles 3D Camera
🎥
722
Adjust camera angles in images using 3D controls or sliders
Running
Featured
4.19k
Wan2.2 Animate
👁
4.19k
Wan2.2 A

In [14]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [15]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [16]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 9 relevant links


'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nCommunity\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nfal/Qwen-Image-Edit-2511-Multiple-Angles-LoRA\nUpdated\n7 days ago\n•\n36.9k\n•\n635\nLightricks/LTX-2\nUpdated\nabout 9 hours ago\n•\n1.06M\n•\n1k\nzai-org/GLM-Image\nUpdated\nabout 22 hours ago\n•\n203\n•\n544\nnvidia/nemotron-speech-streaming-en-0.6b\nUpdated\n9 days ago\n•\n3.9k\n•\n369\nopenbmb/AgentCPM-Explore\nUpdated\nabout 19 hours ago\n•\n77\n•\n266\nBrowse 2M+ models\nSpaces\nRun

In [17]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [18]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 7 relevant links


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


# Hugging Face Brochure

---

## About Hugging Face

Hugging Face is the AI community building the future by providing the **collaboration platform** for the global machine learning community. It serves as a central hub where anyone—from researchers and engineers to enterprises—can share, explore, and experiment with open-source machine learning (ML) models, datasets, and applications.

With over **2 million models** and **500,000+ datasets**, Hugging Face empowers ML engineers, scientists, and end-users to learn, collaborate, and innovate, helping to drive an open and ethical AI future.

---

## What Hugging Face Offers

### The Collaboration Platform
- Host unlimited public ML models, datasets, and apps.
- Build your ML portfolio by sharing work with the global community.
- Support for all modalities: text, image, video, audio, and even 3D data.
- Access a fast-growing community and trending models updated daily.

### Enterprise Solutions
Hugging Face provides tailored options for organizations to scale AI initiatives with:
- **Enterprise-grade security & access controls** including Single Sign-On (SSO).
- Granular management with resource groups, audit logs, and token management.
- Analytics dashboard to monitor repository usage and spending.
- Advanced compute options including ZeroGPU to boost performance.
- Private dataset viewing and additional private storage.
- Priority support and flexible billing options for teams and enterprises.

Pricing starts at $20 per user/month for team subscriptions with custom contracts available for enterprise clients.

---

## Company Culture & Community

Hugging Face fosters a vibrant, **open-source-first culture** centered around collaboration and transparency. The company is dedicated to:
- Building an inclusive AI community that shares knowledge freely.
- Promoting ethical AI development and use.
- Enabling fast-paced innovation by providing accessible and powerful tools.
- Empowering the next generation of ML practitioners globally.

The community includes enthusiastic contributors, researchers, developers, and enterprises that regularly update models, datasets, and host innovative AI applications known as "Spaces."

---

## Careers at Hugging Face

Joining Hugging Face means becoming part of a forward-thinking team on the cutting edge of AI research and engineering. The company values creativity, collaboration, and continuous learning, offering roles in:

- Machine Learning Engineering
- Research and Development
- Software Engineering
- Community Management
- Enterprise Solutions and Customer Success

By working at Hugging Face, you contribute directly to building the tools and partnerships that enable the AI community worldwide.

---

## Why Choose Hugging Face?

- Access the **world’s leading AI collaboration platform**.
- Collaborate with a thriving global AI community.
- Leverage enterprise-grade tools and secure environments.
- Drive innovation with powerful, ready-to-use ML models and datasets.
- Build your own AI applications or enhance existing ones with ease.
- Be part of an ethical, open-source movement shaping the future of AI.

---

## Get Started

- Explore millions of models and datasets at [huggingface.co](https://huggingface.co)
- Try AI applications on Spaces to see ML in action.
- Join the community, share your own work, and build your profile.
- Contact sales to scale AI across your organization with enterprise solutions.

Hugging Face — **The AI community building the future.**

---

**Contact & Links:**

Website: [huggingface.co](https://huggingface.co)  
Enterprise: [Enterprise Hub](https://huggingface.co/enterprise)  
Community & Docs: Available on the website  
Careers: Check the website for current openings and application info

---

*Brand Colors:*  
- Yellow: #FFD21E  
- Orange: #FF9D00  
- Grey: #6B7280

---

This brochure offers a concise overview of Hugging Face’s mission, offerings, community spirit, and career opportunities, designed to inform customers, investors, and prospective recruits about this innovative AI-driven company.

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [19]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [20]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 9 relevant links


# Hugging Face Brochure

---

## About Hugging Face

**Hugging Face** is a vibrant AI community dedicated to building the future of machine learning through open collaboration. As the premier platform where machine learning engineers, scientists, and enthusiasts converge, Hugging Face offers a space to create, discover, share, and experiment with machine learning models, datasets, and applications across multiple modalities — including text, images, video, audio, and even 3D.

---

## Our Platform

- **Model Hub:** Access and browse over 2 million models contributed by a global community, updated regularly. From text generation to image editing and audio processing, our diverse model ecosystem fuels innovation and practical AI applications.

- **Datasets:** Explore a vast library of over 500,000 datasets to power your machine learning projects, with resources constantly being updated and expanded.

- **Spaces:** Showcase and deploy your AI apps with an easy-to-use hosting environment tailored for public machine learning applications, fostering community engagement and collaboration.

- **Open Source Stack:** Leverage a robust, open-source toolkit designed to accelerate model development and deployment, enabling faster iteration and innovation.

- **Enterprise Solutions:** Tailored services for businesses seeking scalable AI solutions while participating in the open AI community.

---

## Community & Culture

At the heart of Hugging Face is a fast-growing, inclusive community focused on openness, ethical AI, and collaboration. We empower the next generation of AI practitioners to build portfolios, learn from peers, and contribute to a global ecosystem of machine learning.

Our mission is to build an **open and ethical AI future together**, ensuring that AI technologies progress in a manner that benefits everyone.

---

## Customers and Users

Hugging Face supports a wide range of users—from individual researchers and developers to large enterprises and academic institutions. Our platform facilitates collaboration on cutting-edge models in natural language processing, computer vision, speech recognition, and more. Our open model hub is one of the most widely used repositories in machine learning worldwide.

---

## Careers & Opportunities

Join Hugging Face to be part of a community-driven company at the forefront of AI innovation. We look for passionate individuals who are eager to contribute to open source, work with state-of-the-art technology, and shape the future of machine learning.

Positions typically include roles in:

- Machine Learning Engineering  
- Research Science  
- Software Development  
- Community Engagement  
- Enterprise Solutions  

If you want to make a difference in the AI landscape by fostering openness and collaboration, Hugging Face offers an inspiring workplace culture to grow and thrive.

---

## Get Involved

- **Explore AI Apps:** Jump into hundreds of AI-powered applications built by the community.  
- **Contribute Models & Datasets:** Share your work to help others accelerate their projects.  
- **Build Your Profile:** Create and showcase your machine learning portfolio.  
- **Sign Up for Compute:** Access paid compute resources to scale your machine learning workloads.

Visit [huggingface.co](https://huggingface.co) to get started today!

---

## Brand Highlights

- Hugging Face’s friendly and approachable brand reflects its community-first values.  
- Signature colors: vibrant yellow (#FFD21E) and orange (#FF9D00), symbolizing creativity and energy.  
- Easily accessible brand assets, including logos in various formats for community use.

---

## Summary

Hugging Face is more than just tech — it’s a movement towards democratizing AI, making machine learning accessible, ethical, and collaborative worldwide. Whether you are a hobbyist, researcher, or enterprise, Hugging Face invites you to join the future of AI together.

---

For more details, visit [huggingface.co](https://huggingface.co).  
Connect with a community shaping the AI frontier!

In [ ]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>